# Feature Engineering

The objective of this notebook is to transform the raw demand data into a machine learning-ready dataset.

Based on the insights obtained during Exploratory Data Analysis (EDA), we will engineer temporal, historical, promotional, and categorical features that help the forecasting model learn demand patterns.

The final output of this notebook will be a processed dataset that will be used for model training.

In [ ]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [ ]:
demand_df = pd.read_csv("../data/raw/demand.csv")
promo_df = pd.read_csv("../data/raw/promotions.csv")

In [ ]:
demand_df["date"] = pd.to_datetime(demand_df["date"])

promo_df["promotion_date"] = pd.to_datetime(
    promo_df["promotion_date"]
)

In [ ]:
print(demand_df.shape)
print(promo_df.shape)

demand_df.head()

In [ ]:
demand_df["demand"] = (
    demand_df
    .groupby(["supermarket", "sku"])["demand"]
    .transform(lambda x: x.fillna(x.median()))
)

In [ ]:
print(demand_df["demand"].isnull().sum())

In [ ]:
promo_df = promo_df.drop(columns=["Unnamed: 0"])

In [ ]:
promo_df["promotion_end"] = (
    promo_df["promotion_date"] +
    pd.Timedelta(days=6)
)

In [ ]:
demand_df["promotion"] = 0

In [ ]:
for _, row in promo_df.iterrows():

    mask = (
        (demand_df["date"] >= row["promotion_date"]) &
        (demand_df["date"] <= row["promotion_end"]) &
        (demand_df["sku"] == row["sku"]) &
        (demand_df["supermarket"] == row["supermarket"])
    )

    demand_df.loc[mask, "promotion"] = 1

In [ ]:
print(demand_df["promotion"].value_counts())

In [ ]:
# Year
demand_df["year"] = demand_df["date"].dt.year

# Month (numeric: 1-12)
demand_df["month"] = demand_df["date"].dt.month

# Quarter (1-4)
demand_df["quarter"] = demand_df["date"].dt.quarter

# Week of year
demand_df["week_of_year"] = demand_df["date"].dt.isocalendar().week.astype(int)

# Day of week (0=Monday, 6=Sunday)
demand_df["day_of_week"] = demand_df["date"].dt.dayofweek

# Day of month
demand_df["day_of_month"] = demand_df["date"].dt.day

# Weekend indicator
demand_df["is_weekend"] = (
    demand_df["day_of_week"] >= 5
).astype(int)

In [ ]:
demand_df.head()

In [ ]:
demand_df = demand_df.sort_values(
    by=["supermarket", "sku", "date"]
).reset_index(drop=True)

In [ ]:
lag_days = [1, 7, 14, 28, 56]

for lag in lag_days:
    demand_df[f"lag_{lag}"] = (
        demand_df
        .groupby(["supermarket", "sku"])["demand"]
        .shift(lag)
    )

In [ ]:
demand_df[
    [
        "date",
        "supermarket",
        "sku",
        "demand",
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
        "lag_56"
    ]
].head(60)

In [ ]:
demand_df["demand_change_1"] = (
    demand_df["demand"] -
    demand_df["lag_1"]
)

demand_df["demand_change_7"] = (
    demand_df["demand"] -
    demand_df["lag_7"]
)

In [ ]:
rolling_windows = [7, 14, 28]

for window in rolling_windows:

    demand_df[f"rolling_mean_{window}"] = (
        demand_df
        .groupby(["supermarket", "sku"])["demand"]
        .transform(
            lambda x: x.shift(1).rolling(window).mean()
        )
    )

    demand_df[f"rolling_std_{window}"] = (
        demand_df
        .groupby(["supermarket", "sku"])["demand"]
        .transform(
            lambda x: x.shift(1).rolling(window).std()
        )
    )

In [ ]:
demand_df[
    [
        "date",
        "demand",
        "rolling_mean_7",
        "rolling_mean_14",
        "rolling_mean_28",
        "rolling_std_7",
        "rolling_std_28"
    ]
].head(40)

In [ ]:
FORECAST_HORIZON = 56

demand_df["target"] = (
    demand_df
    .groupby(["supermarket", "sku"])["demand"]
    .shift(-FORECAST_HORIZON)
)

In [ ]:
demand_df[
    [
        "date",
        "demand",
        "target"
    ]
].tail(65)

In [ ]:
final_df = demand_df.dropna().reset_index(drop=True)

In [ ]:
print(final_df.shape)

print(final_df.isnull().sum())

In [ ]:
from sklearn.preprocessing import LabelEncoder

sku_encoder = LabelEncoder()
market_encoder = LabelEncoder()

final_df["sku"] = sku_encoder.fit_transform(final_df["sku"])

final_df["supermarket"] = market_encoder.fit_transform(final_df["supermarket"])

In [ ]:
import pickle

with open("../models/sku_encoder.pkl", "wb") as f:
    pickle.dump(sku_encoder, f)

with open("../models/supermarket_encoder.pkl", "wb") as f:
    pickle.dump(market_encoder, f)

In [ ]:
final_df.head()

In [ ]:
final_df.tail()

In [ ]:
final_df["target"].describe()

In [ ]:
final_df.to_csv(
    "../data/processed/final_dataset.csv",
    index=False
)

In [ ]:
FEATURES = [
    col for col in final_df.columns
    if col != "target"
]

TARGET = "target"

print(FEATURES)

In [ ]:
X = final_df[FEATURES]
y = final_df[TARGET]

# Feature Engineering Summary

The raw demand dataset was transformed into a machine learning-ready dataset by engineering temporal, historical, promotional, and statistical features.

## The following preprocessing steps were performed:

* Missing demand values were imputed using the median demand within each supermarket-SKU combination.
* Promotion periods were converted into a binary promotion indicator.
* Calendar-based features including year, month, quarter, week of year, day of week, day of month, and weekend indicator were created.
* Historical demand information was captured using lag features at 1, 7, 14, 28, and 56-day intervals.
* Rolling mean and rolling standard deviation features were generated using historical demand while avoiding data leakage through shifted rolling windows.
* The forecasting target was created by shifting demand 56 days into the future for each supermarket-SKU combination.
* Categorical variables were label encoded to prepare the dataset for tree-based machine learning models.

The final processed dataset contains 8,847 observations with 24 engineered features and is ready for model training.